## Import libraries

In [ ]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import precision_score,recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

## Data processing

In [18]:
df = pd.read_excel("Telco Churn dataset.xlsx", engine="openpyxl")
df.isnull().sum()
# df["Totalcalls sum"]= df[["TotalDayCalls","TotalEveCalls","TotalNightCalls","TotalIntlCalls"]].sum(axis=1) # sum the colums to test an idea
# df["Totalcalls sum"].equals(df["TotalCall"]) # compared the two colums
df=df.drop(columns=["customerID", "TotalCall","PaperlessBilling","PaymentMethod"])
x=df.drop(columns=["Churn"])
y=df["Churn"]
y=pd.get_dummies(y,dtype=int)
y

,No,Yes
0,1,0
1,0,1
2,1,0
3,1,0
4,1,0
...,...,...
3328,1,0
3329,0,1
3330,1,0
3331,1,0


## Splitting Data to train and test set

In [ ]:
x_train,x_test,y_train,y_test =train_test_split(x,y,random_state=42,test_size=0.3)

## Label Encoding And Standardization

In [ ]:
cat_cols = x.select_dtypes(include="object").columns.tolist()
num_cols = x.select_dtypes(exclude="object").columns.tolist()
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

## XGboost Classifier

In [ ]:
xg_pipline=Pipeline([("perp",preprocess),("xg",XGBClassifier(subsample=0.8,colsample_bytree=0.8,objective='reg:squarederror',random_state=0))])
param_grid={'xg__n_estimators':[100,150,200,250,300],"xg__learning_rate":[0.05,0.1,0.15,0.2,0.25],"xg__max_depth":[5,6,7,8,9,10]}
xg_search=GridSearchCV(xg_pipline,param_grid,cv=5,scoring="recall",n_jobs=-1)
xg_search.fit(x_train,y_train)
xg_pred=xg_search.predict(x_test)

## Decision Tree

In [ ]:
dt_pipline =Pipeline([("prep",preprocess),("dt",DecisionTreeClassifier())])
param_grid ={"dt__max_depth":[None,10,15,20,30,25]}
dt_search=GridSearchCV(dt_pipline,param_grid,cv=5,scoring="recall",n_jobs=-1)
dt_search.fit(x_train,y_train)
dt_pred=dt_search.predict(x_test)

## Random Forest

In [ ]:
rf_pipline=Pipeline([("prep",preprocess),("rf",RandomForestClassifier())])
param_grid={"rf__n_estimators":[100,150,200,350,300]}
rf_search=GridSearchCV(rf_pipline,param_grid,cv=5,scoring="recall",n_jobs=-1)
rf_search.fit(x_train,y_train)
rf_pred=rf_search.predict(x_test)